# Day 072 — Exercise 2: extract_segments

**What you'll build:** `extract_segments(result) -> list[dict]` — convert Whisper's raw segment list to clean dicts with `{start, end, text, confidence}`.

**Why it matters:** Timestamped segments are the basis of subtitles, jump-to-moment navigation, quality filtering, and segment search.

In [ ]:
_MOCK_RESULT = {
    'text': ' Hello world. This is a test of speech recognition.',
    'language': 'en',
    'segments': [
        {'id': 0, 'start': 0.0, 'end': 3.2, 'text': ' Hello world.',
         'avg_logprob': -0.25, 'no_speech_prob': 0.01},
        {'id': 1, 'start': 3.2, 'end': 7.8,
         'text': ' This is a test of speech recognition.',
         'avg_logprob': -0.30, 'no_speech_prob': 0.02},
    ],
}
_mock_transcribe = lambda source: _MOCK_RESULT


## Task

Implement `extract_segments(result) -> list[dict]`:

For each `seg` in `result.get('segments', [])`:
- `logprob = seg.get('avg_logprob', -1.0)`
- `confidence = min(1.0, max(0.0, 1.0 + logprob))`
- Append `{'start': float(seg.get('start', 0.0)), 'end': float(seg.get('end', 0.0)), 'text': seg.get('text', '').strip(), 'confidence': round(confidence, 4)}`

## Your Implementation

In [ ]:
def extract_segments(result: dict) -> list:
    """Extract time-stamped segments from a whisper result.

    Returns:
        list of dicts: {start: float, end: float, text: str, confidence: float}
        confidence = min(1.0, max(0.0, 1.0 + avg_logprob))
    """
    raise NotImplementedError


In [ ]:
def extract_segments(result):
    out = []
    for seg in result.get('segments', []):
        logprob = seg.get('avg_logprob', -1.0)
        confidence = min(1.0, max(0.0, 1.0 + logprob))
        out.append({
            'start':      float(seg.get('start', 0.0)),
            'end':        float(seg.get('end', 0.0)),
            'text':       seg.get('text', '').strip(),
            'confidence': round(confidence, 4),
        })
    return out


## Automated checks

In [ ]:

score, total = 0, 5
try:
    segs = extract_segments(_MOCK_RESULT)

    # returns a list
    assert isinstance(segs, list) and len(segs) == 2
    score += 1; print("✅ returns list with correct item count")

    # required keys
    assert all('start' in s and 'end' in s and 'text' in s and 'confidence' in s
               for s in segs)
    score += 1; print("✅ all required keys present")

    # start/end are floats
    assert isinstance(segs[0]['start'], float) and isinstance(segs[0]['end'], float)
    assert segs[0]['start'] == 0.0 and abs(segs[0]['end'] - 3.2) < 0.001
    score += 1; print("✅ start/end are correct floats")

    # text is stripped
    assert segs[0]['text'] == 'Hello world.', f"Got {segs[0]['text']!r}"
    score += 1; print("✅ text is stripped")

    # confidence: avg_logprob=-0.25 -> confidence=0.75
    assert abs(segs[0]['confidence'] - 0.75) < 0.001, f"Got {segs[0]['confidence']}"
    # confidence clamped: logprob=-2.0 -> 0.0
    low_seg = {'start': 0.0, 'end': 1.0, 'text': 'noise', 'avg_logprob': -2.0, 'no_speech_prob': 0.9}
    low_result = {'text': 'noise', 'language': 'en', 'segments': [low_seg]}
    low = extract_segments(low_result)
    assert low[0]['confidence'] == 0.0, f"Expected 0.0, got {low[0]['confidence']}"
    score += 1; print("✅ confidence: 1+logprob correct; clamped to 0.0 for logprob<=-1")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def extract_segments(result):
    out = []
    for seg in result.get('segments', []):
        logprob = seg.get('avg_logprob', -1.0)
        confidence = min(1.0, max(0.0, 1.0 + logprob))
        out.append({
            'start':      float(seg.get('start', 0.0)),
            'end':        float(seg.get('end', 0.0)),
            'text':       seg.get('text', '').strip(),
            'confidence': round(confidence, 4),
        })
    return out
```

**Why `min(1.0, max(0.0, ...))`?** avg_logprob can be anywhere in (-inf, 0]. Values below -1 produce negative confidence — we clamp to 0.0. Values very close to 0 produce values above 1 only if logprob > 0, which Whisper never produces, but clamping to 1.0 is defensive.

</details>